# Multi-Texture Scale Experiment - Google Colab GPU Version

**Features:**
- GPU accelerated training
- Batch processing for faster inference
- Table per texture + Final average table

**Instructions:**
1. Runtime > Change runtime type > GPU
2. Run all cells

In [34]:
# Cell 1: Install dependencies (run once)
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q

In [43]:
# Cell 2: Imports and Configuration
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.data import Data, Batch
import os
import pandas as pd
from tqdm.auto import tqdm
import random
import time

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Configuration
# Get the directory where this notebook is located
import pathlib
NOTEBOOK_DIR = pathlib.Path().absolute()
BASE_DIR = os.path.join(str(NOTEBOOK_DIR), 'Scale_Results')
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Results will be saved to: {BASE_DIR}")

# Resolution reduction factor (K=4 means 4x smaller in each dimension)
K = 4

# Original dimensions divided by K
ORIGINAL_X, ORIGINAL_Y, ORIGINAL_Z = 681 // K, 344 // K, max(12 // K, 3)

SCALES = [
    ('original', 1, 1, 1), ('2x1y1z', 2, 1, 1), ('1x2y1z', 1, 2, 1),
    ('1x1y2z', 1, 1, 2), ('2x2y1z', 2, 2, 1), ('1x2y2z', 1, 2, 2),
    ('2x2y2z', 2, 2, 2), ('3x1y1z', 3, 1, 1), ('3x2y1z', 3, 2, 1),
    ('3x1y2z', 3, 1, 2), ('3x2y2z', 3, 2, 2)
]

TEXTURES = ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
# Adjusted parameters for lower resolution
TOP_K, Z_FIXED, MAX_RADIUS, STEP_SIZE = 100, max(5 // K, 1), max(10 // K, 3), max(5 // K, 2)

# Match tolerance: consider 3 voxels difference as a match
MATCH_TOLERANCE = 1

# ============================================================
# EUCLIDEAN DISTANCE NORMALIZATION PARAMETER
# ============================================================
# All Euclidean distances are divided by this value
# Example: if EUCLIDEAN_STEP = 3 and distance = 9, then normalized = 9/3 = 3
EUCLIDEAN_STEP = 1  # voxels - change this to scale the distances

print(f"\nResolution reduction factor: K={K}")
print(f"Match tolerance: {MATCH_TOLERANCE} voxels")
print(f"Euclidean step (normalization): {EUCLIDEAN_STEP} (all distances divided by this)")
print(f"Effective dimensions: {ORIGINAL_X}x{ORIGINAL_Y}x{ORIGINAL_Z}")
print(f"Textures: {TEXTURES}")
print(f"Scales: {len(SCALES)}")
print(f"Total experiments: {len(TEXTURES) * len(SCALES)}")

Device: cpu
Results will be saved to: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results

Resolution reduction factor: K=4
Match tolerance: 1 voxels
Euclidean step (normalization): 1 (all distances divided by this)
Effective dimensions: 170x86x3
Textures: ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
Scales: 11
Total experiments: 55


In [46]:
# Cell 3: Texture Generation Functions (Vectorized)

def generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Sinusoidal wave patterns - vectorized for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    # Scale parameters based on dimensions
    y_center = y_dim // 3
    strip_size = max(y_dim // 15, 3)
    
    # Scale sine parameters proportionally
    sine_params = [
        (0, y_dim // 3, x_dim),      # amplitude, period scaled
        (1, y_dim // 6, x_dim // 2),
        (2, y_dim // 2, x_dim * 3 // 4)
    ]
    
    x_range = np.arange(x_dim)
    for channel, amplitude, period in sine_params:
        if period == 0:
            period = 1
        y_sine = y_center + amplitude * np.sin(2 * np.pi * x_range / period)
        for z in range(z_dim):
            for x in range(x_dim):
                y_c = int(round(y_sine[x]))
                y_start, y_end = max(0, y_c - strip_size//2), min(y_dim, y_c + strip_size//2 + 1)
                for y in range(y_start, y_end):
                    data[channel, np.random.randint(0, 11), z, y, x] = 1
    
    # Channel 3: horizontal line
    y_start, y_end = max(0, y_center - strip_size//2), min(y_dim, y_center + strip_size//2 + 1)
    for z in range(z_dim):
        for x in range(x_dim):
            for y in range(y_start, y_end):
                data[3, np.random.randint(0, 11), z, y, x] = 1
    return data


def generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Linear strip patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    
    # Scale parameters for low resolution
    base_offset = x_dim // 18
    base_width = max(x_dim // 140, 2)
    
    for i in range(3):
        for z in range(z_dim):
            x_start = max(0, int((base_offset + base_offset*i) * x_scale))
            x_end = min(x_dim, int((base_offset + base_width + base_offset*i) * x_scale))
            for y in range(y_dim):
                for x in range(x_start, x_end):
                    data[0, np.random.randint(6, 11), z, y, x] = 1
            
            y_base = y_dim // 20
            y_start = max(0, int((y_base + y_base*i) * y_scale))
            y_end = min(y_dim, int((y_base + base_width + y_base*i) * y_scale))
            for y in range(y_start, y_end):
                for x in range(x_dim):
                    data[1, np.random.randint(6, 11), z, y, x] = 1
            
            strip_w = max(int(2 * max(x_scale, y_scale)), 1)
            diag_offset = x_dim // 12
            for c in [int(-diag_offset * x_scale), 0]:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y - x - c) <= strip_w:
                            data[2, np.random.randint(6, 11), z, y, x] = 1
            for d in [int(diag_offset * y_scale), int(2 * diag_offset * y_scale)]:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y + x - d) <= strip_w:
                            data[3, np.random.randint(6, 11), z, y, x] = 1
    return data


def generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Olympic rings pattern - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    
    # Scale circles based on dimensions
    circles = [
        (0, x_dim * 3 // 10, y_dim * 4 // 10, min(x_dim, y_dim) // 4),
        (1, x_dim * 6 // 10, y_dim * 4 // 10, min(x_dim, y_dim) // 4),
        (2, x_dim * 45 // 100, y_dim * 7 // 10, min(x_dim, y_dim) // 4),
        (3, x_dim * 5 // 10, y_dim * 9 // 10, min(x_dim, y_dim) // 3)
    ]
    stripe_width = max(min(x_dim, y_dim) // 20, 2)
    
    for channel, cx, cy, radius in circles:
        inner_r, outer_r = max(radius - stripe_width, 1), radius
        for z in range(z_dim):
            for y in range(y_dim):
                for x in range(x_dim):
                    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
                    if inner_r <= dist <= outer_r:
                        data[channel, np.random.randint(0, 11), z, y, x] = 1
    return data


def generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Oval/ellipse patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    cx, cy = x_dim // 2, y_dim // 2
    x_stretch = 1.5
    
    # Scale radii based on dimensions
    min_dim = min(x_dim, y_dim)
    inner_r1, outer_r1 = min_dim // 4, min_dim // 2
    inner_r2, outer_r2 = min_dim * 4 // 10, min_dim * 55 // 100
    
    for z in range(z_dim):
        for y in range(y_dim):
            for x in range(x_dim):
                dx, dy = (x - cx) / x_stretch, y - cy
                dist = np.sqrt(dx**2 + dy**2)
                if inner_r1 <= dist <= outer_r1:
                    data[0, np.random.randint(6, 11), z, y, x] = 1
                if inner_r2 <= dist <= outer_r2:
                    data[1, np.random.randint(6, 11), z, y, x] = 1
    
    z_center = z_dim // 2
    cloud_r = (inner_r1 + outer_r1) // 2
    cloud_spread = max(min_dim // 10, 2)
    z_spread = max(z_dim // 4, 1)
    
    for _ in range(5):
        angle = np.random.uniform(0, 2*np.pi)
        r = np.random.uniform(cloud_r * 0.9, cloud_r * 1.1)
        cloud_cx = int(np.clip(cx + r * x_stretch * np.cos(angle), cloud_spread, x_dim - cloud_spread - 1))
        cloud_cy = int(np.clip(cy + r * np.sin(angle), cloud_spread, y_dim - cloud_spread - 1))
        for _ in range(max(20, min_dim // 5)):
            rx = np.random.randint(-cloud_spread, cloud_spread + 1)
            ry = np.random.randint(-cloud_spread, cloud_spread + 1)
            rz = np.random.randint(-z_spread, z_spread + 1)
            px, py, pz = cloud_cx + rx, cloud_cy + ry, z_center + rz
            if 0 <= px < x_dim and 0 <= py < y_dim and 0 <= pz < z_dim:
                data[2, np.random.randint(6, 11), pz, py, px] = 1
                data[3, np.random.randint(6, 11), pz, py, px] = 1
    return data


def generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Colony cloud patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    min_dim = min(x_dim, y_dim)
    max_radius = max(min_dim // 8, 3)
    cloud_points = max(10, min_dim // 10)
    
    def add_cloud(channel, cx, cy, z, radius):
        for _ in range(cloud_points):
            angle = np.random.uniform(0, 2*np.pi)
            r = np.random.uniform(0, radius)
            px = int(np.clip(cx + r*np.cos(angle), 0, x_dim-1))
            py = int(np.clip(cy + r*np.sin(angle), 0, y_dim-1))
            data[channel, np.random.randint(6, 11), z, py, px] = 1
    
    # Scale step sizes
    arc_steps = max(x_dim // 8, 6)
    sine_step = max(x_dim // 8, 4)
    diag_step = max(x_dim // 6, 5)
    
    for z in range(z_dim):
        for t in np.linspace(0, 1, arc_steps):
            arc_x = int(t * (x_dim - 1))
            arc_y = int(t * (y_dim - 1) + 0.3 * (y_dim - 1) * np.sin(t * np.pi))
            arc_y = int(np.clip(arc_y, 0, y_dim-1))
            add_cloud(0, arc_x, arc_y, z, np.random.uniform(max_radius // 3, max_radius))
        
        for x in range(0, x_dim, sine_step):
            y_center = y_dim // 3
            y_amp = y_dim // 6
            y = int(y_center + y_amp * np.sin(2*np.pi*x/x_dim))
            y = int(np.clip(y, 0, y_dim-1))
            add_cloud(1, x, y, z, np.random.uniform(max_radius // 3, max_radius))
        
        diag_offsets = [y_dim // 5, y_dim * 2 // 5]
        for d in diag_offsets:
            for x in range(0, x_dim, diag_step):
                y = -x + int(d * y_scale)
                if 0 <= y < y_dim:
                    add_cloud(3, x, y, z, np.random.uniform(max_radius // 3, max_radius))
    return data


def create_groundtruth(texture_type, scale_name, x_scale, y_scale, z_scale, output_dir):
    """Create ground truth with specified texture and scale"""
    num_channels, num_values = 4, 11
    x_dim = ORIGINAL_X * x_scale
    y_dim = ORIGINAL_Y * y_scale  
    z_dim = ORIGINAL_Z * z_scale
    local_x_scale, local_y_scale = x_dim / 172, y_dim / 87
    
    if texture_type == 'sinusoid':
        data = generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'linear':
        data = generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    elif texture_type == 'olympic':
        data = generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'oval':
        data = generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'colonies':
        data = generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    else:
        raise ValueError(f"Unknown texture: {texture_type}")
    
    filepath = os.path.join(output_dir, f'groundtruth_{texture_type}_{scale_name}.npy')
    np.save(filepath, data)
    return data, filepath

print("Texture generation functions loaded!")

Texture generation functions loaded!


In [48]:
# Cell 4: Subgraph Creation (Optimized)

def create_subgraphs(data, texture_type, scale_name, output_dir):
    """Fast subgraph creation using vectorized operations"""
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    z_idx = min(Z_FIXED, z_dim - 1)
    
    # Pre-compute mask
    intensity_matrix = np.zeros((y_dim, x_dim, num_channels), dtype=np.float32)
    channel_mask = np.zeros((y_dim, x_dim, num_channels), dtype=bool)
    
    for ch in range(num_channels):
        ch_data = data[ch, :, z_idx, :, :]
        mask = ch_data.sum(axis=0) > 0
        intensity_matrix[:, :, ch] = np.where(mask, np.argmax(ch_data, axis=0), 0)
        channel_mask[:, :, ch] = mask
    
    channel_counts = channel_mask.sum(axis=2)
    centers = [(x, y, z_idx) for x in range(0, x_dim, STEP_SIZE) for y in range(0, y_dim, STEP_SIZE)]
    
    all_subgraphs = []
    for cx, cy, cz in centers:
        x_min, x_max = max(0, cx - MAX_RADIUS), min(x_dim, cx + MAX_RADIUS + 1)
        y_min, y_max = max(0, cy - MAX_RADIUS), min(y_dim, cy + MAX_RADIUS + 1)
        
        nodes, positions, active_chs = [], [], []
        for y in range(y_min, y_max):
            for x in range(x_min, x_max):
                if channel_counts[y, x] > 0:
                    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
                    if dist <= MAX_RADIUS:
                        active = np.where(channel_mask[y, x, :])[0].tolist()
                        nodes.append(intensity_matrix[y, x, active])
                        positions.append((x, y, z_idx))
                        active_chs.append(active)
        
        if len(nodes) < 2:
            continue
            
        max_ch = max(len(ch) for ch in active_chs)
        padded = []
        for i, n in enumerate(nodes):
            p = np.zeros(max_ch, dtype=np.float32)
            p[:len(n)] = n
            padded.append(p)
        
        node_features = np.array(padded, dtype=np.float32)
        pos_array = np.array(positions, dtype=np.int32)
        
        # Fast edge creation
        diff = pos_array[:, np.newaxis, :] - pos_array[np.newaxis, :, :]
        dist_matrix = np.sqrt(np.sum(diff**2, axis=2))
        edge_mask = (dist_matrix <= MAX_RADIUS) & (dist_matrix > 0)
        edge_i, edge_j = np.where(edge_mask)
        
        if len(edge_i) == 0:
            continue
        
        edge_weights = dist_matrix[edge_i, edge_j].astype(np.float32)
        
        graph = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=torch.tensor([edge_i, edge_j], dtype=torch.long),
            edge_attr=torch.tensor(edge_weights, dtype=torch.float32),
            center=(cx, cy, cz)
        )
        all_subgraphs.append(graph)
    
    return all_subgraphs

print("Subgraph creation function loaded!")

Subgraph creation function loaded!


In [49]:
# Cell 5: Model Definition (GPU Optimized)

class ContrastiveGAT(nn.Module):
    def __init__(self, in_channels=4, hidden=32, proj_dim=16, heads=4, dropout=0.1, edge_dim=None):
        super().__init__()
        self.edge_dim = edge_dim
        kw = dict(dropout=dropout)
        if edge_dim: kw['edge_dim'] = edge_dim
        
        self.gat1 = GATConv(in_channels, hidden, heads=heads, concat=True, **kw)
        self.gat2 = GATConv(hidden*heads, hidden, heads=heads, concat=True, **kw)
        self.gat3 = GATConv(hidden*heads, hidden, heads=1, concat=False, **kw)
        
        self.norm1 = nn.LayerNorm(hidden*heads)
        self.norm2 = nn.LayerNorm(hidden*heads)
        self.dropout = nn.Dropout(dropout)
        
        self.projection = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, proj_dim)
        )
        self.interaction_head = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden//2, 1)
        )
    
    def forward(self, x, edge_index, edge_attr=None, batch=None):
        ea = edge_attr if self.edge_dim and edge_attr is not None else None
        x = F.elu(self.dropout(self.norm1(self.gat1(x, edge_index, ea))))
        x = F.elu(self.dropout(self.norm2(self.gat2(x, edge_index, ea))))
        x = F.elu(self.gat3(x, edge_index, ea))
        
        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
        
        emb = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch), global_add_pool(x, batch)], dim=1)
        proj = F.normalize(self.projection(emb), dim=1)
        score = self.interaction_head(emb)
        return proj, score


def prepare_graph(graph, target_channels=4):
    x = graph.x.clone()
    if x.shape[1] < target_channels:
        x = torch.cat([x, torch.zeros(x.shape[0], target_channels - x.shape[1])], dim=1)
    elif x.shape[1] > target_channels:
        x = x[:, :target_channels]
    g = Data(x=x, edge_index=graph.edge_index.clone())
    if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
        g.edge_attr = graph.edge_attr.clone()
    if hasattr(graph, 'gt_score'):
        g.gt_score = graph.gt_score
    return g


def graph_augment(graph, target_channels=4):
    g = prepare_graph(graph, target_channels)
    num_nodes = g.x.shape[0]
    mask_n = int(num_nodes * 0.1)
    if mask_n > 0:
        g.x[torch.randperm(num_nodes)[:mask_n]] = 0.0
    return g


def contrastive_loss(z1, z2, temp=0.1):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    sim = torch.matmul(z1, z2.T) / temp
    labels = torch.arange(z1.shape[0], device=z1.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2


def train_model_gpu(model, graphs, device, epochs=1, batch_size=32, lr=0.01, gradient_accumulation_steps=4):
    """GPU optimized training with ranking loss for better top-k prediction"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    target_ch = max(g.x.shape[1] for g in graphs) if graphs else 4
    
    print(f"  Training: {epochs} epochs, batch_size={batch_size}, lr={lr}")
    
    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()
        shuffled_graphs = graphs.copy()
        random.shuffle(shuffled_graphs)
        
        for batch_idx, i in enumerate(range(0, len(shuffled_graphs), batch_size)):
            batch_g = shuffled_graphs[i:i+batch_size]
            aug1 = [graph_augment(g, target_ch) for g in batch_g]
            aug2 = [graph_augment(g, target_ch) for g in batch_g]
            
            # Move to GPU
            for g in aug1 + aug2:
                g.x, g.edge_index = g.x.to(device), g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)
            
            try:
                b1, b2 = Batch.from_data_list(aug1), Batch.from_data_list(aug2)
            except:
                continue
            
            z1, p1 = model(b1.x, b1.edge_index, getattr(b1, 'edge_attr', None), b1.batch)
            z2, _ = model(b2.x, b2.edge_index, getattr(b2, 'edge_attr', None), b2.batch)
            
            # Contrastive loss
            loss_contrast = contrastive_loss(z1, z2)
            
            # Supervised losses on gt_score
            loss_reg = torch.tensor(0.0, device=device)
            loss_rank = torch.tensor(0.0, device=device)
            
            if hasattr(b1, 'gt_score'):
                try:
                    gt = b1.gt_score.to(device).float()
                    pred = p1.view(-1)
                    
                    # 1. MSE Regression loss (normalized)
                    if gt.std() > 0:
                        gt_norm = (gt - gt.mean()) / (gt.std() + 1e-8)
                    else:
                        gt_norm = gt
                    if pred.std() > 0:
                        pred_norm = (pred - pred.mean()) / (pred.std() + 1e-8)
                    else:
                        pred_norm = pred
                    loss_reg = F.mse_loss(pred_norm, gt_norm)
                    
                    # 2. Pairwise Ranking loss - learn correct ordering
                    # For pairs where gt[i] > gt[j], we want pred[i] > pred[j]
                    if len(gt) >= 2:
                        n_pairs = min(len(gt) * (len(gt) - 1) // 2, 100)  # Limit pairs
                        idx = torch.randperm(len(gt))[:min(20, len(gt))]  # Sample indices
                        loss_rank_sum = 0.0
                        n_valid = 0
                        for ii in range(len(idx)):
                            for jj in range(ii + 1, len(idx)):
                                i_idx, j_idx = idx[ii], idx[jj]
                                if gt[i_idx] > gt[j_idx]:
                                    # pred[i] should be > pred[j], margin = 0.1
                                    loss_rank_sum += F.relu(0.1 - (pred[i_idx] - pred[j_idx]))
                                    n_valid += 1
                                elif gt[i_idx] < gt[j_idx]:
                                    loss_rank_sum += F.relu(0.1 - (pred[j_idx] - pred[i_idx]))
                                    n_valid += 1
                        if n_valid > 0:
                            loss_rank = loss_rank_sum / n_valid
                except:
                    pass
            
            # Combined loss: contrastive + regression + ranking
            # Increase supervised weight for better ranking
            loss = (loss_contrast + 1.0 * loss_reg + 0.5 * loss_rank) / gradient_accumulation_steps
            loss.backward()
            
            epoch_losses.append(loss.item() * gradient_accumulation_steps)
            
            # Update weights every gradient_accumulation_steps
            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            # Cleanup
            del z1, z2, b1, b2, aug1, aug2
            if device.type == 'cuda':
                torch.cuda.empty_cache()
        
        # Final update for remaining batches
        if len(shuffled_graphs) // batch_size % gradient_accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        avg_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")
    
    return model

print("Model and training functions loaded!")

Model and training functions loaded!


In [50]:
# Cell 6: Coordinate Extraction & Accuracy (GPU Batch Processing)

def attach_gt_scores(data, graphs):
    """Compute GT score as: sum of (intensity + channel_count) for each voxel within radius.
    
    For each voxel:
        - channel_count = number of channels with any nonzero value
        - intensity = sum of value indices (0-10) across all active channels
        - voxel_score = channel_count + intensity
    
    Graph gt_score = sum of voxel_scores within MAX_RADIUS of center
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    
    # Pre-compute per-voxel scores
    # channel_active[c, z, y, x] = True if channel c has any value > 0
    channel_active = (data > 0).any(axis=1)  # (C, Z, Y, X)
    channel_count = channel_active.sum(axis=0).astype(np.float32)  # (Z, Y, X)
    
    # Compute intensity sum across channels
    intensity_sum = np.zeros((z_dim, y_dim, x_dim), dtype=np.float32)
    for ch in range(num_channels):
        ch_data = data[ch, :, :, :, :]  # (V, Z, Y, X)
        intensity_per_voxel = np.argmax(ch_data, axis=0).astype(np.float32)  # (Z, Y, X)
        intensity_per_voxel = intensity_per_voxel * channel_active[ch].astype(np.float32)
        intensity_sum += intensity_per_voxel
    
    # Combined voxel score
    voxel_score = channel_count + intensity_sum  # (Z, Y, X)
    
    for g in graphs:
        cx, cy, cz = int(g.center[0]), int(g.center[1]), int(g.center[2])
        cz = np.clip(cz, 0, z_dim-1)
        
        x_min, x_max = max(0, cx-MAX_RADIUS), min(x_dim, cx+MAX_RADIUS+1)
        y_min, y_max = max(0, cy-MAX_RADIUS), min(y_dim, cy+MAX_RADIUS+1)
        
        yy, xx = np.meshgrid(np.arange(y_min, y_max), np.arange(x_min, x_max), indexing='ij')
        dist_sq = (xx - cx)**2 + (yy - cy)**2
        radius_mask = dist_sq <= MAX_RADIUS**2
        
        # Sum voxel scores within radius
        scores_in_radius = voxel_score[cz, y_min:y_max, x_min:x_max]
        g.gt_score = float((scores_in_radius * radius_mask).sum())
    
    return graphs


def find_top_positions_batch(model, graphs, device, top_k=100, batch_size=128):
    """Batch GPU inference for faster processing"""
    model.eval()
    target_ch = max(g.x.shape[1] for g in graphs) if graphs else 4
    all_scores = []
    
    with torch.no_grad():
        for i in range(0, len(graphs), batch_size):
            batch_g = graphs[i:i+batch_size]
            prepared = [prepare_graph(g, target_ch) for g in batch_g]
            
            for g in prepared:
                g.x, g.edge_index = g.x.to(device), g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)
            
            try:
                batch = Batch.from_data_list(prepared)
                _, scores = model(batch.x, batch.edge_index, getattr(batch, 'edge_attr', None), batch.batch)
                scores = scores.cpu().numpy().flatten()
                
                for j, g in enumerate(batch_g):
                    all_scores.append({
                        'x': g.center[0], 'y': g.center[1], 'z': g.center[2], 
                        'score': float(scores[j])
                    })
            except:
                for g in batch_g:
                    all_scores.append({'x': g.center[0], 'y': g.center[1], 'z': g.center[2], 'score': 0.0})
    
    all_scores.sort(key=lambda s: s['score'], reverse=True)
    return all_scores[:top_k]


def extract_gt_coords(data, graphs, top_k=100):
    """Extract Ground Truth top-k coordinates based on intensity + channel count.
    
    Algorithm:
        For each graph center (cx, cy, cz):
            - For each voxel within radius MAX_RADIUS:
                - channel_count = number of channels with any nonzero value
                - intensity = sum of value indices (0-10) across active channels
                - voxel_score = channel_count + intensity
            - graph_score = sum of voxel_scores within radius
        
        Sort by graph_score descending and return top_k centers.
    
    This uses the SAME scoring as attach_gt_scores for consistency.
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    
    # Pre-compute per-voxel scores (same as attach_gt_scores)
    channel_active = (data > 0).any(axis=1)  # (C, Z, Y, X)
    channel_count = channel_active.sum(axis=0).astype(np.float32)  # (Z, Y, X)
    
    intensity_sum = np.zeros((z_dim, y_dim, x_dim), dtype=np.float32)
    for ch in range(num_channels):
        ch_data = data[ch, :, :, :, :]  # (V, Z, Y, X)
        intensity_per_voxel = np.argmax(ch_data, axis=0).astype(np.float32)
        intensity_per_voxel = intensity_per_voxel * channel_active[ch].astype(np.float32)
        intensity_sum += intensity_per_voxel
    
    voxel_score = channel_count + intensity_sum  # (Z, Y, X)
    
    all_scores = []
    radius_sq = MAX_RADIUS ** 2
    
    for graph in graphs:
        cx, cy, cz = graph.center
        cx, cy, cz = int(cx), int(cy), int(cz)
        cz = max(0, min(z_dim - 1, cz))
        
        x_min = max(0, cx - MAX_RADIUS)
        x_max = min(x_dim, cx + MAX_RADIUS + 1)
        y_min = max(0, cy - MAX_RADIUS)
        y_max = min(y_dim, cy + MAX_RADIUS + 1)
        
        # Sum voxel scores within radius
        score = 0.0
        for y in range(y_min, y_max):
            dy2 = (y - cy) ** 2
            for x in range(x_min, x_max):
                dx2 = (x - cx) ** 2
                if dx2 + dy2 <= radius_sq:
                    score += voxel_score[cz, y, x]
        
        all_scores.append({
            'x': cx,
            'y': cy,
            'z': cz,
            'score': score
        })
    
    # Sort by score descending
    all_scores.sort(key=lambda s: s['score'], reverse=True)
    return all_scores[:top_k]


def normalize_euclidean(distances):
    """Normalize Euclidean distances by dividing by EUCLIDEAN_STEP.
    
    Uses global parameter:
        EUCLIDEAN_STEP: all distances are divided by this value
    
    Example: if EUCLIDEAN_STEP = 3 and distance = 9, then normalized = 9/3 = 3
    
    Args:
        distances: Array or list of Euclidean distances
    
    Returns:
        Normalized distances array (distances / EUCLIDEAN_STEP)
    """
    distances = np.array(distances)
    # Simply divide all distances by EUCLIDEAN_STEP
    distances = distances / EUCLIDEAN_STEP
    return distances


def compute_accuracy(model_coords, gt_coords, tol=None, outlier_percentile=10):
    """Compute Euclidean distances between model TOP_K and GT TOP_K (one-to-one pairing).
    
    Algorithm:
        - Model gives TOP_K points (e.g., 100), GT has TOP_K points (e.g., 100)
        - For each model point (in order), find the closest UNUSED GT point
        - Compute Euclidean distance for this pair
        - Remove top 10% outliers (highest distances) before computing statistics
        - Statistics (mean, std, min, max) are computed over remaining pairs
    
    Args:
        model_coords: List of model predicted coordinates (TOP_K points)
        gt_coords: List of ground truth coordinates (TOP_K points)
        tol: Match tolerance in voxels (for counting "matches" only)
        outlier_percentile: Remove top X% of distances as outliers (default: 10)
    
    Returns:
        Dictionary with Euclidean distance statistics (after outlier removal)
        (Euclidean distances are normalized by dividing by EUCLIDEAN_STEP)
    """
    if tol is None:
        tol = MATCH_TOLERANCE
    
    if not model_coords or not gt_coords:
        return {'matches': 0, 
                'model_count': len(model_coords), 'gt_count': len(gt_coords),
                'euclidean_mean': 0, 'euclidean_std': 0, 'euclidean_min': 0, 'euclidean_max': 0,
                'pairwise_distances': []}
    
    model_pts = np.array([[c['x'], c['y'], c['z']] for c in model_coords])
    gt_pts = np.array([[c['x'], c['y'], c['z']] for c in gt_coords])
    
    # Compute pairwise Euclidean distances matrix
    diff = model_pts[:, np.newaxis, :] - gt_pts[np.newaxis, :, :]
    dist = np.sqrt(np.sum(diff**2, axis=2))  # Shape: (n_model, n_gt)
    
    # One-to-one matching: for each model point, find closest unmatched GT point
    n_model = len(model_pts)
    used_gt = set()
    pairwise_distances = []  # Store Euclidean distance for each of the TOP_K pairs
    matches = 0  # Count of pairs within tolerance
    
    for i in range(n_model):
        # Find closest GT that hasn't been used yet
        sorted_gt_indices = np.argsort(dist[i])
        for j in sorted_gt_indices:
            if j not in used_gt:
                # Pair model point i with GT point j
                distance = dist[i, j]
                pairwise_distances.append(distance)
                used_gt.add(j)
                if distance <= tol:
                    matches += 1
                break
        else:
            # No more GT points available (rare case)
            pairwise_distances.append(np.nan)
    
    m_count, g_count = len(model_pts), len(gt_pts)
    
    # Convert to numpy array and remove NaN values
    pairwise_distances = np.array(pairwise_distances)
    valid_distances = pairwise_distances[~np.isnan(pairwise_distances)]
    
    if len(valid_distances) > 0:
        # Remove top 10% outliers (highest distances)
        # Calculate the percentile threshold
        percentile_threshold = np.percentile(valid_distances, 100 - outlier_percentile)
        # Keep only distances below the threshold
        filtered_distances = valid_distances[valid_distances <= percentile_threshold]
        
        if len(filtered_distances) > 0:
            # Apply normalization (divide by EUCLIDEAN_STEP)
            normalized_distances = normalize_euclidean(filtered_distances)
            euclidean_mean = np.mean(normalized_distances)
            euclidean_std = np.std(normalized_distances)
            euclidean_min = np.min(normalized_distances)
            euclidean_max = np.max(normalized_distances)
            n_removed = len(valid_distances) - len(filtered_distances)
        else:
            euclidean_mean = euclidean_std = euclidean_min = euclidean_max = 0.0
            normalized_distances = []
            n_removed = len(valid_distances)
    else:
        euclidean_mean = euclidean_std = euclidean_min = euclidean_max = 0.0
        normalized_distances = []
        n_removed = 0
    
    return {
        'matches': matches,  # Number of pairs within tolerance
        'model_count': m_count, 
        'gt_count': g_count,
        'tolerance': tol,
        'outliers_removed': n_removed,  # Number of outliers removed (top 10%)
        # Normalized Euclidean distances after outlier removal
        'euclidean_mean': euclidean_mean,
        'euclidean_std': euclidean_std,
        'euclidean_min': euclidean_min,
        'euclidean_max': euclidean_max,
        'pairwise_distances': list(normalized_distances) if len(normalized_distances) > 0 else []
    }

print("Coordinate extraction and accuracy functions loaded!")

Coordinate extraction and accuracy functions loaded!


In [52]:
# Cell 7: Main Execution - Run All Experiments

def print_table(texture_name, results):
    """Print formatted table for one texture with Euclidean distances (all TOP_K pairs)"""
    print(f"\n{'='*80}")
    print(f"RESULTS TABLE: {texture_name.upper()} (Euclidean over all TOP_K pairs)")
    print(f"{'='*80}")
    print(f"{'Scale Name':>12} {'Scale':>12} {'Matches':>8} {'Euc Mean':>12} {'Euc Std':>10} {'Euc Min':>10} {'Euc Max':>10}")
    print("-"*80)
    for r in results:
        print(f"{r['Scale Name']:>12} {r['Scale']:>12} {r['Matches']:>8} {r['Euc Mean']:>12.4f} {r['Euc Std']:>10.4f} {r['Euc Min']:>10.4f} {r['Euc Max']:>10.4f}")
    print("="*80)


def run_texture(texture_type, device):
    """Run all scales for one texture"""
    print(f"\n{'*'*80}")
    print(f"TEXTURE: {texture_type.upper()}")
    print(f"{'*'*80}")
    
    texture_dir = os.path.join(BASE_DIR, texture_type)
    os.makedirs(texture_dir, exist_ok=True)
    
    results = []
    
    for scale_name, x_s, y_s, z_s in tqdm(SCALES, desc=f"{texture_type}"):
        try:
            # 1. Create ground truth
            data, gt_path = create_groundtruth(texture_type, scale_name, x_s, y_s, z_s, texture_dir)
            
            # 2. Create subgraphs
            graphs = create_subgraphs(data, texture_type, scale_name, texture_dir)
            
            # 3. Filter graphs (use reasonable min_nodes based on K)
            # Original code uses 100, but with K=4 we need fewer
            MIN_NODES = max(10, 100 // (K * K))
            filtered = [g for g in graphs if g.x.shape[0] >= MIN_NODES]
            
            if not filtered:
                print(f"  WARNING {scale_name}: No graphs passed filter (total: {len(graphs)}, min_nodes={MIN_NODES})")
                # Try with lower threshold
                MIN_NODES = 2
                filtered = [g for g in graphs if g.x.shape[0] >= MIN_NODES]
                if not filtered:
                    continue
            
            print(f"  {scale_name}: {len(filtered)} graphs after filter (min_nodes={MIN_NODES})")
            
            # 4. Attach GT scores to graphs for supervised training
            filtered = attach_gt_scores(data, filtered)
            
            # 5. Train model (GPU) - 10 epochs like original
            in_ch = max(g.x.shape[1] for g in filtered)
            edge_dim = 1 if any(hasattr(g, 'edge_attr') and g.edge_attr is not None for g in filtered) else None
            model = ContrastiveGAT(in_ch, 32, 16, 4, 0.1, edge_dim).to(device)
            
            # Subsample for training if too many graphs
            train_graphs = filtered[::2] if len(filtered) > 20000 else filtered
            model = train_model_gpu(model, train_graphs, device, epochs=1, batch_size=32, lr=0.01, gradient_accumulation_steps=4)
            
            # 6. Extract model predictions (batch GPU)
            model_coords = find_top_positions_batch(model, filtered, device, TOP_K, batch_size=128)
            
            # 7. Extract GT coordinates
            gt_coords = extract_gt_coords(data, filtered, TOP_K)
            
            # 8. Compute accuracy with MAX_RADIUS as tolerance (same as original code)
            acc = compute_accuracy(model_coords, gt_coords, MAX_RADIUS)
            
            results.append({
                'Scale Name': scale_name,
                'Scale': f'{x_s}x,{y_s}x,{z_s}x',
                'Matches': acc['matches'],
                'Tolerance': MAX_RADIUS,
                # Euclidean distance metrics for matched pairs
                'Euc Mean': acc['euclidean_mean'],
                'Euc Std': acc['euclidean_std'],
                'Euc Min': acc['euclidean_min'],
                'Euc Max': acc['euclidean_max']
            })
            
            # Cleanup
            del data, graphs, filtered, model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"  ERROR {scale_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Print table
    if results:
        print_table(texture_type, results)
        df = pd.DataFrame(results)
        csv_path = os.path.join(texture_dir, f'results_{texture_type}.csv')
        df.to_csv(csv_path, index=False)
        print(f"  Saved: {csv_path}")
    else:
        print(f"  WARNING: No results for {texture_type}")
    
    return results

print("Execution functions loaded!")

Execution functions loaded!


In [53]:
# Cell 8: RUN ALL EXPERIMENTS

print("="*80)
print("MULTI-TEXTURE SCALE EXPERIMENT - GPU VERSION")
print("="*80)
print(f"\nDevice: {device}")
print(f"Textures: {TEXTURES}")
print(f"Total experiments: {len(TEXTURES) * len(SCALES)}")

start_time = time.time()
all_results = {}

# Run each texture
for texture in TEXTURES:
    all_results[texture] = run_texture(texture, device)

total_time = time.time() - start_time
print(f"\nTotal time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")

MULTI-TEXTURE SCALE EXPERIMENT - GPU VERSION

Device: cpu
Textures: ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
Total experiments: 55

********************************************************************************
TEXTURE: SINUSOID
********************************************************************************


sinusoid:   0%|          | 0/11 [00:00<?, ?it/s]

  original: 976 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 4.186947


sinusoid:   9%|▉         | 1/11 [00:06<01:08,  6.86s/it]

  2x1y1z: 1862 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 3.534977


...

  3x1y2z: 2092 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 2.687962


oval:  91%|█████████ | 10/11 [08:22<00:51, 51.99s/it]

  3x2y2z: 8110 graphs after filter (min_nodes=10)
  Training: 1 epochs, batch_size=32, lr=0.01
    Epoch 1/1 - Loss: 2.646164


oval: 100%|██████████| 11/11 [09:53<00:00, 53.95s/it]


RESULTS TABLE: OVAL (Euclidean over all TOP_K pairs)
  Scale Name        Scale  Matches     Euc Mean    Euc Std    Euc Min    Euc Max
--------------------------------------------------------------------------------
    original     1x,1x,1x       29      11.2426    14.3884     0.0000    59.4643
      2x1y1z     2x,1x,1x       52       7.9243    13.9349     0.0000    66.2118
      1x2y1z     1x,2x,1x       21      22.5280    28.5976     0.0000   138.1304
      1x1y2z     1x,1x,2x       26       9.8443     8.2098     0.0000    31.1127
      2x2y1z     2x,2x,1x       15      34.9995    43.1165     0.0000   145.9863
      1x2y2z     1x,2x,2x        9      21.9780    32.6025     2.0000   130.3840
      2x2y2z     2x,2x,2x       15      21.9195    21.1351     0.0000    85.2760
      3x1y1z     3x,1x,1x       55       4.5956     6.3414     0.0000    25.2982
      3x2y1z     3x,2x,1x       18      28.1550    36.3123     0.0000   136.0147
      3x1y2z     3x,1x,2x       36       9.0268    10.6

In [55]:
# Cell 9: Final Summary Tables & Save to CSV

# Use all 5 textures - outlier removal happens per scale (1 max removed per scale)
EXCLUDE_LAST_TEXTURE = False  # Use all 5 textures
TEXTURES_FOR_SUMMARY = TEXTURES[:-1] if EXCLUDE_LAST_TEXTURE else TEXTURES
print(f"Textures included in summary: {TEXTURES_FOR_SUMMARY}")
print(f"Total textures: {len(TEXTURES_FOR_SUMMARY)}")
print(f"For Average by Scale: 1 max outlier removed per scale -> {len(TEXTURES_FOR_SUMMARY) - 1} textures averaged")

# Collect all results (excluding last texture if specified)
combined = []
for tex, res_list in all_results.items():
    if EXCLUDE_LAST_TEXTURE and tex == TEXTURES[-1]:
        continue  # Skip last texture
    for r in res_list:
        r_copy = r.copy()
        r_copy['Texture'] = tex
        combined.append(r_copy)

if combined:
    # ============================================================
    # 1. Save each texture results to separate CSV
    # ============================================================
    print(f"\n{'='*80}")
    print("SAVING INDIVIDUAL TEXTURE RESULTS")
    print(f"{'='*80}")
    
    for tex in TEXTURES_FOR_SUMMARY:
        tex_data = [r for r in combined if r['Texture'] == tex]
        if tex_data:
            tex_df = pd.DataFrame(tex_data)
            tex_csv_path = os.path.join(BASE_DIR, f'results_{tex}.csv')
            tex_df.to_csv(tex_csv_path, index=False)
            print(f"  Saved: {tex_csv_path} ({len(tex_data)} rows)")
    
    # ============================================================
    # 2. Create Average by Texture table and save to CSV (Euclidean only)
    # ============================================================
    print(f"\n{'='*100}")
    print("AVERAGE BY TEXTURE - EUCLIDEAN DISTANCE (across all scales)")
    print(f"{'='*100}")
    print(f"{'Texture':>12} {'Experiments':>12} {'Avg Matches':>12} {'Avg Euc Mean':>14} {'Avg Euc Std':>12} {'Avg Euc Min':>12} {'Avg Euc Max':>12}")
    print("-"*100)
    
    avg_by_texture = []
    for tex in TEXTURES_FOR_SUMMARY:
        tex_data = [r for r in combined if r['Texture'] == tex]
        if tex_data:
            avg_row = {
                'Texture': tex,
                'Experiments': len(tex_data),
                'Avg Matches': np.mean([r['Matches'] for r in tex_data]),
                # Euclidean distance averages only
                'Avg Euc Mean': np.mean([r['Euc Mean'] for r in tex_data]),
                'Avg Euc Std': np.mean([r['Euc Std'] for r in tex_data]),
                'Avg Euc Min': np.mean([r['Euc Min'] for r in tex_data]),
                'Avg Euc Max': np.mean([r['Euc Max'] for r in tex_data])
            }
            avg_by_texture.append(avg_row)
            print(f"{tex:>12} {avg_row['Experiments']:>12} {avg_row['Avg Matches']:>12.2f} {avg_row['Avg Euc Mean']:>14.4f} {avg_row['Avg Euc Std']:>12.4f} {avg_row['Avg Euc Min']:>12.4f} {avg_row['Avg Euc Max']:>12.4f}")
    
    print("="*80)
    
    # Save average by texture
    avg_texture_df = pd.DataFrame(avg_by_texture)
    avg_texture_csv = os.path.join(BASE_DIR, 'average_by_texture.csv')
    avg_texture_df.to_csv(avg_texture_csv, index=False)
    print(f"  Saved: {avg_texture_csv}")
    
    # ============================================================
    # 3. Create Average by Scale table (remove 1 max outlier per scale)
    # ============================================================
    print(f"\n{'='*120}")
    print("AVERAGE BY SCALE - EUCLIDEAN DISTANCE (remove 1 max Euc Mean per scale)")
    print(f"{'='*120}")
    print(f"{'Scale':>12} {'Config':>14} {'Textures':>10} {'Removed':>10} {'Avg Matches':>12} {'Avg Euc Mean':>14} {'Avg Euc Std':>12} {'Avg Euc Min':>12} {'Avg Euc Max':>12}")
    print("-"*120)
    
    avg_by_scale = []
    for scale_name, x_s, y_s, z_s in SCALES:
        scale_data = [r for r in combined if r['Scale Name'] == scale_name]
        if scale_data and len(scale_data) > 1:
            # Sort by Euc Mean descending and remove the max (first one)
            sorted_data = sorted(scale_data, key=lambda r: r['Euc Mean'], reverse=True)
            removed_texture = sorted_data[0]['Texture']  # The one with highest Euc Mean
            filtered_data = sorted_data[1:]  # Remove the max
            
            avg_row = {
                'Scale Name': scale_name,
                'Config': f'{x_s}x,{y_s}y,{z_s}z',
                'Textures Used': len(filtered_data),
                'Removed Texture': removed_texture,
                'Avg Matches': np.mean([r['Matches'] for r in filtered_data]),
                # Euclidean distance averages (after removing max)
                'Avg Euc Mean': np.mean([r['Euc Mean'] for r in filtered_data]),
                'Avg Euc Std': np.mean([r['Euc Std'] for r in filtered_data]),
                'Avg Euc Min': np.mean([r['Euc Min'] for r in filtered_data]),
                'Avg Euc Max': np.mean([r['Euc Max'] for r in filtered_data])
            }
            avg_by_scale.append(avg_row)
            print(f"{scale_name:>12} {avg_row['Config']:>14} {avg_row['Textures Used']:>10} {removed_texture:>10} {avg_row['Avg Matches']:>12.2f} {avg_row['Avg Euc Mean']:>14.4f} {avg_row['Avg Euc Std']:>12.4f} {avg_row['Avg Euc Min']:>12.4f} {avg_row['Avg Euc Max']:>12.4f}")
        elif scale_data:
            # Only 1 texture, can't remove any
            avg_row = {
                'Scale Name': scale_name,
                'Config': f'{x_s}x,{y_s}y,{z_s}z',
                'Textures Used': len(scale_data),
                'Removed Texture': 'none',
                'Avg Matches': np.mean([r['Matches'] for r in scale_data]),
                'Avg Euc Mean': np.mean([r['Euc Mean'] for r in scale_data]),
                'Avg Euc Std': np.mean([r['Euc Std'] for r in scale_data]),
                'Avg Euc Min': np.mean([r['Euc Min'] for r in scale_data]),
                'Avg Euc Max': np.mean([r['Euc Max'] for r in scale_data])
            }
            avg_by_scale.append(avg_row)
            print(f"{scale_name:>12} {avg_row['Config']:>14} {avg_row['Textures Used']:>10} {'none':>10} {avg_row['Avg Matches']:>12.2f} {avg_row['Avg Euc Mean']:>14.4f} {avg_row['Avg Euc Std']:>12.4f} {avg_row['Avg Euc Min']:>12.4f} {avg_row['Avg Euc Max']:>12.4f}")
    
    print("="*80)
    
    # Save average by scale
    avg_scale_df = pd.DataFrame(avg_by_scale)
    avg_scale_csv = os.path.join(BASE_DIR, 'average_by_scale.csv')
    avg_scale_df.to_csv(avg_scale_csv, index=False)
    print(f"  Saved: {avg_scale_csv}")
    
    # ============================================================
    # 4. Overall Average and save to CSV
    # ============================================================
    print(f"\n{'='*80}")
    print("OVERALL AVERAGE - EUCLIDEAN DISTANCE")
    print(f"{'='*80}")
    
    overall_avg = {
        'Total Experiments': len(combined),
        'Overall Avg Matches': np.mean([r['Matches'] for r in combined]),
        # Euclidean distance averages only
        'Overall Avg Euc Mean': np.mean([r['Euc Mean'] for r in combined]),
        'Overall Avg Euc Std': np.mean([r['Euc Std'] for r in combined]),
        'Overall Avg Euc Min': np.mean([r['Euc Min'] for r in combined]),
        'Overall Avg Euc Max': np.mean([r['Euc Max'] for r in combined])
    }
    
    print(f"  Total Experiments: {overall_avg['Total Experiments']}")
    print(f"  Overall Avg Matches: {overall_avg['Overall Avg Matches']:.2f}")
    print(f"  -------- Euclidean Distance (all TOP_K one-to-one pairs) --------")
    print(f"  Overall Avg Euclidean Mean: {overall_avg['Overall Avg Euc Mean']:.4f}")
    print(f"  Overall Avg Euclidean Std: {overall_avg['Overall Avg Euc Std']:.4f}")
    print(f"  Overall Avg Euclidean Min: {overall_avg['Overall Avg Euc Min']:.4f}")
    print(f"  Overall Avg Euclidean Max: {overall_avg['Overall Avg Euc Max']:.4f}")
    print("="*80)
    
    # Save overall average
    overall_df = pd.DataFrame([overall_avg])
    overall_csv = os.path.join(BASE_DIR, 'overall_average.csv')
    overall_df.to_csv(overall_csv, index=False)
    print(f"  Saved: {overall_csv}")
    
    # ============================================================
    # 5. Save all combined results
    # ============================================================
    all_results_csv = os.path.join(BASE_DIR, 'all_textures_all_scales_results.csv')
    pd.DataFrame(combined).to_csv(all_results_csv, index=False)
    print(f"  Saved: {all_results_csv}")

# ============================================================
# Summary of saved files
# ============================================================
# print(f"\n{'='*80}")
# print("ALL CSV FILES SAVED:")
# print(f"{'='*80}")
# print(f"  Directory: {BASE_DIR}")
# print(f"  - results_<texture>.csv  : Results for each texture (5 files)")
# print(f"  - average_by_texture.csv : Average metrics per texture")
# print(f"  - average_by_scale.csv   : Average metrics per scale")
# print(f"  - overall_average.csv    : Overall average metrics")
# # print(f"  - all_textures_all_scales_results.csv : All raw results")
# print(f"{'='*80}")
# print("EXPERIMENT COMPLETE!")
# print(f"{'='*80}")

Textures included in summary: ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']
Total textures: 5
For Average by Scale: 1 max outlier removed per scale -> 4 textures averaged

SAVING INDIVIDUAL TEXTURE RESULTS
  Saved: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results\results_sinusoid.csv (11 rows)
  Saved: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results\results_colonies.csv (11 rows)
  Saved: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results\results_linear.csv (11 rows)
  Saved: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results\results_olympic.csv (11 rows)
  Saved: d:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\Scale_Results\results_oval.csv (11 rows)

AVERAGE BY TEXTURE - EUCLIDEAN DISTANCE (across all scales)
     Texture  Experiments  Avg Matches   Avg Euc Mean  Avg Euc Std  Avg Euc Min  Avg Euc Max
----------------------------------------------------------------------------------------------------
    sinusoid           11        76.73

   Scale         Config  Textures    Removed   Avg Matches   Avg Euc Mean  ...
original      1x,1y,1z         3   colonies          85.33         1.2345  ...
  2x1y1z      2x,1y,1z         3   sinusoid          72.00         2.1234  ...